In [ ]:
"""
Moduł interaktywnego panelu sterowania do wizualizacji rotacji brył 3D
z wykorzystaniem akceleracji sprzętowej na platformie FPGA Kria KV260.

Skrypt definiuje podstawowe modele przestrzenne (sześcian, ostrosłup, walec),
zarządza alokacją pamięci CMA, obsługuje transfery danych przez AXI DMA
oraz buduje interaktywny interfejs oparty na bibliotece ipywidgets i matplotlib.
"""

import numpy as np
import math
import time
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pynq import Overlay, allocate
from IPython.display import display, HTML
import ipywidgets as widgets

print("[INIT] Konfiguracja sprzętu FPGA...")
overlay = Overlay("design_1_wrapper.xsa")
dma = overlay.axi_dma_0

cube_vertices = [
    ( 50,  50,  50), ( 50, -50,  50), (-50, -50,  50), (-50,  50,  50),
    ( 50,  50, -50), ( 50, -50, -50), (-50, -50, -50), (-50,  50, -50)
]
cube_edges = [[0,1], [1,2], [2,3], [3,0], [4,5], [5,6], [6,7], [7,4], [0,4], [1,5], [2,6], [3,7]]

pyramid_vertices = [
    ( 50,  50, -50), ( 50, -50, -50), (-50, -50, -50), (-50,  50, -50),
    (  0,   0,  60)
]
pyramid_edges = [[0,1], [1,2], [2,3], [3,0], [0,4], [1,4], [2,4], [3,4]]

cylinder_bottom = [(int(50*math.cos(i*math.pi/4)), int(50*math.sin(i*math.pi/4)), -50) for i in range(8)]
cylinder_top    = [(int(50*math.cos(i*math.pi/4)), int(50*math.sin(i*math.pi/4)), 50) for i in range(8)]
cylinder_vertices = cylinder_bottom + cylinder_top
cylinder_edges = [[i, (i+1)%8] for i in range(8)] + [[i+8, (i+1)%8 + 8] for i in range(8)] + [[i, i+8] for i in range(8)]

MAX_VERTICES = 16
tx_buffer = allocate(shape=(4 * MAX_VERTICES,), dtype=np.int32)
rx_buffer = allocate(shape=(3 * MAX_VERTICES,), dtype=np.int32)

def hw_rotate_3d(current_vertices, angle_deg, axis):
    """
    Wykonuje sprzętowy obrót zbioru wierzchołków wokół wskazanej osi układu 3D,
    komunikując się z potokiem sprzętowym FPGA za pośrednictwem DMA.

    Args:
        current_vertices (list of tuples): Lista krotek (x, y, z) z obecnymi współrzędnymi.
        angle_deg (float): Kąt obrotu w stopniach.
        axis (str): Oś obrotu, przyjmuje wartości 'X', 'Y' lub 'Z'.

    Returns:
        list of tuples: Nowa lista zaktualizowanych współrzędnych po rotacji.
    """
    normalized_angle = ((angle_deg + 180) % 360) - 180
    invert_coords = False

    if normalized_angle > 90:
        normalized_angle -= 180
        invert_coords = True
    elif normalized_angle < -90:
        normalized_angle += 180
        invert_coords = True

    angle_hw = int(math.radians(normalized_angle) * 1024)
    num_v = len(current_vertices)

    for i, (x, y, z) in enumerate(current_vertices):
        if axis == 'Z':
            in1, in2, in3 = x, y, z
        elif axis == 'X':
            in1, in2, in3 = y, z, x
        elif axis == 'Y':
            in1, in2, in3 = z, x, y

        if invert_coords:
            in1, in2 = -in1, -in2

        tx_buffer[i*4+0] = angle_hw
        tx_buffer[i*4+1] = int(in1)
        tx_buffer[i*4+2] = int(in2)
        tx_buffer[i*4+3] = int(in3)

    tx_buffer.flush()

    dma.recvchannel.transfer(rx_buffer[0 : num_v*3])
    dma.sendchannel.transfer(tx_buffer[0 : num_v*4])
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    rx_buffer.invalidate()

    new_vertices = []
    for i in range(num_v):
        out1, out2, out3 = rx_buffer[i*3+0], rx_buffer[i*3+1], rx_buffer[i*3+2]
        if axis == 'Z': nx, ny, nz = out1, out2, out3
        elif axis == 'X': nx, ny, nz = out3, out1, out2
        elif axis == 'Y': nx, ny, nz = out2, out3, out1
        new_vertices.append((nx, ny, nz))

    return new_vertices

def create_shape_ui(title_text, base_vertices, base_edges):
    """
    Konstruuje autonomiczny blok widżetów (UI) zawierający elementy sterujące
    (suwaki) i widok przestrzenny (wykres 3D) dla pojedynczej figury.

    Args:
        title_text (str): Tekstowy tytuł modułu widżetu oraz wykresu.
        base_vertices (list of tuples): Referencyjne położenie wierzchołków.
        base_edges (list of lists): Połączenia definiujące krawędzie.

    Returns:
        ipywidgets.widgets.HBox: Kompletny kontener poziomy z wyrenderowanym panelem sterowania.
    """
    cmap = plt.get_cmap('tab20')

    layout = widgets.Layout(width='300px')
    slider_x = widgets.IntSlider(min=-360, max=360, step=5, value=0, description='Obrót X [°]', layout=layout, continuous_update=False)
    slider_y = widgets.IntSlider(min=-360, max=360, step=5, value=0, description='Obrót Y [°]', layout=layout, continuous_update=False)
    slider_z = widgets.IntSlider(min=-360, max=360, step=5, value=0, description='Obrót Z [°]', layout=layout, continuous_update=False)

    def update_3d_plot(kat_x, kat_y, kat_z):
        """
        Funkcja zwrotna wywoływana przy każdej zmianie suwaka. Renderuje
        wykres 3D w oparciu o sekwencyjny obrót figury wokół wszystkich 3 osi.
        """
        state_hw = base_vertices.copy()
        if kat_x != 0: state_hw = hw_rotate_3d(state_hw, kat_x, 'X')
        if kat_y != 0: state_hw = hw_rotate_3d(state_hw, kat_y, 'Y')
        if kat_z != 0: state_hw = hw_rotate_3d(state_hw, kat_z, 'Z')

        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(111, projection='3d')
        ax.view_init(elev=30, azim=-60)
        v_arr = np.array(state_hw)

        ax.plot([-100, 100], [0, 0], [0, 0], color='red', linestyle='--', alpha=0.3)
        ax.plot([0, 0], [-100, 100], [0, 0], color='green', linestyle='--', alpha=0.3)
        ax.plot([0, 0], [0, 0], [-100, 100], color='blue', linestyle='--', alpha=0.3)

        for edge in base_edges:
            p1, p2 = v_arr[edge[0]], v_arr[edge[1]]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color='black', linestyle='-', linewidth=1.5, alpha=0.8)

        for i, (x, y, z) in enumerate(v_arr):
            color = cmap(i % 20)
            ax.scatter(x, y, z, color=color, s=50, edgecolors='black', zorder=5)

        ax.set_xlim([-100, 100]); ax.set_ylim([-100, 100]); ax.set_zlim([-100, 100])
        ax.set_xlabel('Oś X'); ax.set_ylabel('Oś Y'); ax.set_zlabel('Oś Z')
        ax.set_title(title_text, fontweight='bold', fontsize=12)
        plt.show()

    out = widgets.interactive_output(update_3d_plot, {'kat_x': slider_x, 'kat_y': slider_y, 'kat_z': slider_z})

    controls = widgets.VBox([widgets.HTML(f"<h3>Sterowanie: {title_text}</h3>"), slider_x, slider_y, slider_z])
    ui_block = widgets.HBox([controls, out], layout=widgets.Layout(align_items='center', border='1px solid #ddd', padding='10px', margin='10px 0'))

    return ui_block

def interactive_dashboard():
    """
    Agreguje zbudowane sekcje (Sześcian, Ostrosłup, Walec) i renderuje
    je w notatniku jako zbiorczy dashboard sprzętowej akceleracji 3D.
    """
    ui_cube = create_shape_ui("Sześcian", cube_vertices, cube_edges)
    ui_pyramid = create_shape_ui("Ostrosłup", pyramid_vertices, pyramid_edges)
    ui_cylinder = create_shape_ui("Walec", cylinder_vertices, cylinder_edges)

    display(widgets.HTML("<h2>System Sprzętowej Akceleracji 3D (Kria KV260)</h2>"))
    display(widgets.VBox([ui_cube, ui_pyramid, ui_cylinder]))

try:
    interactive_dashboard()
finally:
    pass